# 01 — Data Audit and Preparation

## Decoding Winger Playstyles

This notebook audits and prepares the historical player dataset used in the project.

The broader project investigates whether players with similar conventional football output can have meaningfully different attacking styles, with a focus on winger comparison for recruitment and role fit.

Before any modelling or player profiling is performed, this notebook establishes:

- dataset structure and coverage,
- column meanings,
- data quality,
- duplicate and redundant variables,
- player identity handling,
- and the correct observational unit of the dataset.

## 1. Dataset

The dataset contains player-level statistics from the five major European domestic leagues between the 2017/18 and 2023/24 seasons.

The data was collected from FBref and published as cleaned seasonal CSV files on Kaggle.

Competitions covered:

- Premier League
- La Liga
- Serie A
- Bundesliga
- Ligue 1

The original Kaggle dataset contains one CSV per season with a consistent schema.

Although the dataset description refers to each row as a "player-season entry", the audit below shows that the practical observational unit is more precisely a **player × club × season** record. Players who transfer during a season can therefore appear more than once.

In [25]:
import pandas as pd
import numpy as np

## 2. Load and combine the seasonal datasets

All seven seasonal CSV files use the same schema. They are loaded individually and concatenated into one dataframe for the remainder of the analysis.

In [26]:
files = [
    "2017-18.csv",
    "2018-19.csv",
    "2019-20.csv",
    "2020-21.csv",
    "2021-22.csv",
    "2022-23.csv",
    "2023-24.csv"
]

dfs = [pd.read_csv(file) for file in files]

In [27]:
reference_columns = dfs[0].columns.tolist()

same_schema = all(
    df_.columns.tolist() == reference_columns
    for df_ in dfs
)

print("Consistent schema across all seasons:", same_schema)

Consistent schema across all seasons: True


In [28]:
df_raw = pd.concat(dfs, ignore_index=True)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

Rows: 18243
Columns: 65


In [29]:
season_counts = (
    df_raw["season"]
    .value_counts()
    .sort_index()
    .rename("rows")
    .to_frame()
)

display(season_counts)

print("Missing values:", df_raw.isna().sum().sum())
print("Exact duplicate rows:", df_raw.duplicated().sum())

,rows
season,
2017-2018,2549
2018-2019,2503
2019-2020,2558
2020-2021,2634
2021-2022,2689
2022-2023,2687
2023-2024,2623


Missing values: 0
Exact duplicate rows: 0


### Initial audit

The seven files use the same 65-column schema and contain 18,243 total records.

No missing values were found in the source files.

Several column names, however, were found to be misleading or inconsistent with the values they contain. These are corrected below before further analysis.

## 3. Correct misleading and inconsistent column names

Several source-column names do not accurately describe their contents.

For example:

- `Avg Mins per Match` contains total seasonal minutes rather than average minutes per match.
- `Shots p 90` corresponds to shots on target per 90 rather than total shots per 90.
- `Times tackled during take-on` contains a percentage rather than a count.
- `1/3` represents passes into the final third.

The dataset also uses inconsistent naming conventions across different FBref tables. These fields are renamed to improve clarity while preserving their underlying values.

In [30]:
df = df_raw.copy()

rename_map = {
    "Avg Mins per Match": "Minutes",
    "Non Penalty Goals": "Non-Penalty Goals",
    "Exp NPG": "Non-Penalty Expected Goals",

    "Goals p 90": "Goals per 90",
    "Assists p 90": "Assists per 90",

    "Tackles attempted": "Tackles Attempted",
    "% Dribbles tackled": "Dribblers Tackled %",

    "Goals against p 90": "Goals Against per 90",
    "Saves %": "Save %",
    "% Clean sheets": "Clean Sheet %",
    "% Penalty saves": "Penalty Save %",

    "Pass completion %": "Pass Completion %",
    "Progressive passes distance": "Progressive Passing Distance",
    "% Short pass completed": "Short Pass Completion %",
    "% Medium passes completed": "Medium Pass Completion %",
    "% Long passes completed": "Long Pass Completion %",

    "1/3": "Passes into Final 3rd",
    "Passes into penalty area": "Passes into Penalty Area",
    "touches_def_pen": "Touches in Defensive Penalty Area",

    "Take ons attempted": "Take-ons Attempted",
    "% Successful take-ons": "Successful Take-ons %",
    "Times tackled during take-on": "Tackled During Take-on %",

    "carries final 3rd": "Carries into Final 3rd",
    "carries penalty area": "Carries into Penalty Area",

    "Possessions lost": "Times Dispossessed",

    "% Shots on target": "Shots on Target %",
    "Shots p 90": "Shots on Target per 90",

    "% Aerial Duels won": "Aerial Duels Won %",
    "Shot creating actions p 90": "Shot Creating Actions per 90",
    "Goal creating actions p 90": "Goal Creating Actions per 90"
}

df = df.rename(columns=rename_map)

> **Data dictionary caveat**
>
> The original field `Possessions lost` appears to represent times a player was dispossessed rather than every possession lost. The Kaggle dataset does not provide a formal field-level data dictionary, so this variable is provisionally renamed `Times Dispossessed`.
>
> It should not be used in important modelling decisions until its interpretation is sufficiently validated.

## 4. Validate derived and duplicate statistics

Several variables appear to either duplicate another field or represent simple arithmetic identities.

These relationships are tested across every row before any columns are removed.

In [31]:
checks = pd.Series({
    "Goals == Goals Scored":
        (df["Goals"] == df["Goals Scored"]).mean(),

    "Progressive Carries == carries_prgc":
        (df["Progressive Carries"] == df["carries_prgc"]).mean(),

    "Non-Penalty Goals == Goals - Penalties":
        (
            df["Non-Penalty Goals"]
            == df["Goals"] - df["Penalty Kicks Made"]
        ).mean(),

    "Goals & Assists == Goals + Assists":
        (
            df["Goals & Assists"]
            == df["Goals"] + df["Assists"]
        ).mean()
}, name="Share of rows passing check")

display(checks.to_frame())

,Share of rows passing check
Goals == Goals Scored,1.0
Progressive Carries == carries_prgc,1.0
Non-Penalty Goals == Goals - Penalties,1.0
Goals & Assists == Goals + Assists,1.0


In [32]:
assert checks.eq(1.0).all()

All four consistency checks hold across 100% of the dataset.

Two columns are exact duplicates:

- `Goals Scored` duplicates `Goals`
- `carries_prgc` duplicates `Progressive Carries`

The dataset also contains `rk`, an original table ranking/index field with no analytical meaning.

These three columns can therefore be safely removed.

In [33]:
df = df.drop(columns=[
    "rk",
    "Goals Scored",
    "carries_prgc"
])

print("Columns after removing redundant fields:", df.shape[1])

Columns after removing redundant fields: 62


## 5. Resolve player identity

Player names cannot be assumed to uniquely identify footballers.

For example, multiple different players in the dataset share names such as `Adama Traoré`, `João Pedro`, `Rodri`, and `Marcelo`.

A provisional player identifier is therefore constructed using:

**player name + birth year**

Birth year is preferred over nationality because nationality can potentially change across a player's career.

In [34]:
df["player_id"] = (
    df["player"].str.strip()
    + "_"
    + df["born"].astype(str)
)

### Validate the identifier

The newly created identifier is tested for cases in which the same name and birth year are associated with multiple nationalities.

This test returned no conflicts.

Manual inspection of repeated player-season records identified one remaining collision: two separate players named Vitinha were both born in 2000.

This case therefore requires a manual identity correction.

In [35]:
player_id_check = (
    df.groupby(["player", "born"])
      .agg(
          nations=("nation", "nunique"),
          squads=("squad", "nunique"),
          seasons=("season", "nunique"),
          total_rows=("player", "size")
      )
      .reset_index()
)

identity_conflicts = player_id_check[
    player_id_check["nations"] > 1
]

print("Name + birth-year IDs with multiple nationalities:",
      len(identity_conflicts))

Name + birth-year IDs with multiple nationalities: 0


In [36]:
vitinha_mask = (
    (df["player"] == "Vitinha")
    & (df["born"] == 2000)
)

df.loc[
    vitinha_mask & (df["pos"] == "MF"),
    "player_id"
] = "vitinha_2000_midfielder"

df.loc[
    vitinha_mask & (df["pos"] == "FW"),
    "player_id"
] = "vitinha_2000_forward"

## 6. Determine the observational unit

A player may appear more than once in a season when they represent multiple clubs.

These are not duplicate records. They represent distinct club spells.

For the purposes of this project, this is useful because a player's role can differ across tactical environments.

The working observational unit is therefore:

**player × club × season**

rather than simply player × season.

In [37]:
player_season_counts = (
    df.groupby(["player_id", "season"])
      .size()
)

multi_club_player_seasons = (
    player_season_counts[
        player_season_counts > 1
    ]
    .sort_values(ascending=False)
)

display(
    multi_club_player_seasons
    .head(20)
    .rename("club_spell_rows")
    .to_frame()
)

,,club_spell_rows
player_id,season,
Fabio Depaoli_1997,2020-2021,3
Maxwel Cornet_1996,2021-2022,2
Maya Yoshida_1988,2019-2020,2
Mehdi Lacen_1984,2017-2018,2
Memphis_1994,2022-2023,2
Mergim Berisha_1998,2023-2024,2
Michael Gregoritsch_1994,2019-2020,2
Michael Krohn-Dehli_1983,2017-2018,2
Michy Batshuayi_1993,2017-2018,2


In [38]:
grain_duplicates = df.duplicated(
    subset=["player_id", "season", "squad", "comp"]
).sum()

print("Duplicate player-club-season records:", grain_duplicates)

Duplicate player-club-season records: 0


## 7. Final sanity checks

Before saving the audited dataset, basic range checks are performed on playing time and percentage variables.

In [39]:
percentage_columns = [
    "Dribblers Tackled %",
    "Save %",
    "Clean Sheet %",
    "Penalty Save %",
    "Pass Completion %",
    "Short Pass Completion %",
    "Medium Pass Completion %",
    "Long Pass Completion %",
    "Successful Take-ons %",
    "Tackled During Take-on %",
    "Shots on Target %",
    "Aerial Duels Won %"
]

percentage_violations = {
    col: (~df[col].between(0, 100)).sum()
    for col in percentage_columns
}

print("Rows with non-positive minutes:",
      (df["Minutes"] <= 0).sum())

print("Rows with non-positive matches:",
      (df["Matches Played"] <= 0).sum())

print("\nPercentage range violations:")
print(percentage_violations)

Rows with non-positive minutes: 0
Rows with non-positive matches: 0

Percentage range violations:
{'Dribblers Tackled %': np.int64(0), 'Save %': np.int64(0), 'Clean Sheet %': np.int64(0), 'Penalty Save %': np.int64(0), 'Pass Completion %': np.int64(0), 'Short Pass Completion %': np.int64(0), 'Medium Pass Completion %': np.int64(0), 'Long Pass Completion %': np.int64(0), 'Successful Take-ons %': np.int64(0), 'Tackled During Take-on %': np.int64(0), 'Shots on Target %': np.int64(0), 'Aerial Duels Won %': np.int64(0)}


## 8. Audit result

After auditing and cleaning:

- all seven seasons use a consistent schema,
- no missing values are present,
- misleading column names have been corrected,
- redundant variables have been removed,
- player-name collisions have been addressed,
- and the dataset's observational unit has been established as player × club × season.

The resulting dataset is now suitable for constructing the winger analysis population and engineering style-related features.

No modelling is performed in this notebook.

In [41]:
print("Final shape:", df.shape)

df

Final shape: (18243, 63)


,player,nation,pos,squad,comp,age,born,Matches Played,Minutes,Goals,...,Shots on Target %,Shots on Target per 90,Goals per shot,Goals per shot on target,Aerial Duels Won %,Shot Creating Actions per 90,Goal Creating Actions per 90,Crosses Stopped,season,player_id
0,Patrick van Aanholt,Netherlands,DF,Crystal Palace,Premier League,26,1990,28,2184,5,...,33.3,0.45,0.15,0.45,54.5,1.90,0.16,0.0,2017-2018,Patrick van Aanholt_1990
1,Rolando Aarons,England,MF,Newcastle Utd,Premier League,21,1995,4,139,0,...,0.0,0.00,0.00,0.00,25.0,0.65,0.00,0.0,2017-2018,Rolando Aarons_1995
2,Rolando Aarons,England,MF,Hellas Verona,Serie A,21,1995,11,517,0,...,0.0,0.00,0.00,0.00,37.5,0.87,0.00,0.0,2017-2018,Rolando Aarons_1995
3,Ignazio Abate,Italy,DF,Milan,Serie A,30,1986,17,1057,1,...,50.0,0.17,0.25,0.50,55.6,2.30,0.34,0.0,2017-2018,Ignazio Abate_1986
4,Aymen Abdennour,Tunisia,DF,Marseille,Ligue 1,27,1989,8,499,0,...,50.0,0.18,0.00,0.00,50.0,0.36,0.00,0.0,2017-2018,Aymen Abdennour_1989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18238,Lovro Zvonarek,Croatia,FW,Bayern Munich,Bundesliga,18,2005,5,163,1,...,100.0,0.55,1.00,1.00,28.6,1.66,0.55,0.0,2023-2024,Lovro Zvonarek_2005
18239,Martin Ødegaard,Norway,MF,Arsenal,Premier League,24,1998,35,3091,8,...,28.0,0.61,0.08,0.29,25.0,6.41,0.67,0.0,2023-2024,Martin Ødegaard_1998
18240,Milan Đurić,Bosnia and Herzegovina,FW,Hellas Verona,Serie A,33,1990,20,1204,5,...,59.1,0.97,0.18,0.31,75.8,2.47,0.30,0.0,2023-2024,Milan Đurić_1990
18241,Milan Đurić,Bosnia and Herzegovina,FW,Monza,Serie A,33,1990,17,1257,4,...,23.1,0.43,0.15,0.67,68.3,0.93,0.07,0.0,2023-2024,Milan Đurić_1990


In [42]:
df.to_csv(
    "fbref_2017_2024_audited.csv",
    index=False
)

In [44]:
df.to_parquet(
    "fbref_2017_2024_audited.parquet",
    index=False
)